In [32]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

In [33]:
pc = Pinecone(api_key=os.getenv("pinecone_api_key"))

In [5]:
pc.create_index(
  name="social-media",
  dimension=768,
  metric="cosine",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  )
)

{
    "name": "social-media",
    "metric": "cosine",
    "host": "social-media-ph2q0am.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "20

In [8]:
index_name = "social-media"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

# 3. Generate and Insert Embeddings
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
text_data = ["This is a test document", "Pinecone stores vectors."]
embeddings = model.encode(text_data).tolist() # Convert to list for Pinecone

# Upsert with IDs and optional metadata
vectors = [
    {"id": f"vec{i}", "values": emb, "metadata": {"text": text}} 
    for i, (emb, text) in enumerate(zip(embeddings, text_data))
]
index.upsert(vectors=vectors)

/media/aumoza/Strg_1/Freelance/personal/Testing_Embeddings_For_recommendation/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6226.25it/s]


UpsertResponse(upserted_count=2, _response_info={'raw_headers': {'date': 'Tue, 28 Apr 2026 17:35:10 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '3151', 'x-pinecone-request-latency-ms': '568', 'x-envoy-upstream-service-time': '259', 'x-pinecone-response-duration-ms': '570', 'grpc-status': '0', 'server': 'envoy'}})

In [18]:
vectors_to_upsert = [
    {
        "id": "item_1003",  # Your specific custom ID
        "values": embeddings[0], 
        "metadata": {
            "text": "Life update"
        }
    },
    {
        "id": "item_1004",
        "values": embeddings[1],
        "metadata": {
            "text": "How come we are not getting any likes on our posts?"
        }
    }
]

index.upsert(vectors=vectors_to_upsert)

UpsertResponse(upserted_count=2, _response_info={'raw_headers': {'date': 'Tue, 28 Apr 2026 17:50:47 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '4', 'x-pinecone-request-logical-size': '3176', 'x-pinecone-request-latency-ms': '472', 'x-envoy-upstream-service-time': '160', 'x-pinecone-response-duration-ms': '474', 'grpc-status': '0', 'server': 'envoy'}})

## Retrieve

In [19]:
query_text = "I am damn too hungry to cook."
query_vector = model.encode(query_text).tolist()

In [20]:
# 2. Query Pinecone for the top matches
results = index.query(
    vector=query_vector,
    top_k=3,                  # Number of related results to get
    include_values=True,      # Returns the raw embedding values
    include_metadata=True     # Returns your custom ID and stored text
)

In [21]:
for match in results['matches']:
    print(f"ID: {match['id']}")
    print(f"Score (Similarity): {match['score']}")
    print(f"Stored Text: {match['metadata']['text']}")

ID: item_1001
Score (Similarity): 0.424156219
Stored Text: Preparing pizza tonight!
ID: vec0
Score (Similarity): 0.424156219
Stored Text: This is a test document
ID: item_1003
Score (Similarity): 0.424156219
Stored Text: Life update
